In [1]:
import pandas as pd  
from pandas import Series, DataFrame 
import uproot 
from scipy import stats
from scipy.optimize import curve_fit
from scipy.special import comb
from scipy.stats import chi2
from scipy.special import comb
from scipy.optimize import lsq_linear
import sys
from plot_tools import *
from customStats import *
#import tools
import common_tools
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
# from selection_cuts import selection_nominal
import mplhep as hep
from sklearn.model_selection import train_test_split
plt.style.use(hep.style.CMS)
plt.rcParams['figure.figsize'] = [10,8]
plt.rcParams['font.size'] = 24
plt.figure()
plt.close()
plt.rcParams.update({'figure.figsize':[10,8]})
plt.rcParams.update({'font.size':24})
import tensorflow as tf
import math
import zfit
from zfit import z
import xgboost as xgb
from scipy.interpolate import make_interp_spline
# from loadCutXGB import load_and_cutXGBclfs
from scipy.special import comb
from scipy.optimize import lsq_linear
zfit.settings.set_verbosity(0)
import json
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # Oculta los mensajes de INFO y WARNING
from utils_efficiency import *
from utils_fits import *


2026-03-16 23:39:56.154154: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-16 23:39:56.180390: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-16 23:39:56.602754: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/ghcp/miniconda3/envs/haza_wokr_env/lib/python3.8/site-packages/zfit/__init__.py:63: UserWarning: TensorFlow warnings are by default suppressed by zfit. In order to show them, set the environment variable ZFIT_DISABLE_TF_WARNINGS=0. In order to suppress th

#  CARGA DE DATOS

In [2]:
import uproot
import pandas as pd

# --- RUTAS DE ARCHIVOS ---
f_gen = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/GenLevel_Angular_Merged.root"
f_gen_filtered = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/GenLevel_Angular_Merged_Filtered.root"
f_reco_gen = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/RecoGenV2_Angular_Merged.root"  
x_gboost_cut = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/BdtoK0smumu-20251110T171511Z-1-001/MyReweiting/ResultsB0_2022/AntiRadVeto_MC_NoRes_2022_Era1_v0_XGBoost_fom_cut_BDT.root"

vars_gen_to_load = ["gen_cosThetaK", "gen_cosThetaL", "gen_phi", "q2Gen"]
vars_reco_to_load = ["CosThetaK_best", "CosThetaL_best", "Phi_best", "massJ"] 
vars_xgboost_to_load = ["CosThetaK", "CosThetaL", "Phi", "massB_test", "massJ", "TotalWeight"] 

# --- CARGA DE DATOS ---
#Gen NO filt
genNFtr = uproot.open(f_gen)['ntuple'].arrays(vars_gen_to_load, library='pd')
print(f"1. Gen Non-Filtered (genNFtr) cargado: {len(genNFtr)} eventos")
# Gen Filtered
genFtr = uproot.open(f_gen_filtered)['ntuple'].arrays(vars_gen_to_load, library='pd')
print(f"2. Gen Filtered (genFtr) cargado: {len(genFtr)} eventos")
# Reco Gen Level
recoGen = uproot.open(f_reco_gen)['ntuple'].arrays(vars_reco_to_load, library='pd')
print(f"3. Reco Gen Level Denom (recoGen) cargado: {len(recoGen)} eventos")
# Final selection 
recoFtr = uproot.open(x_gboost_cut)['treeBd'].arrays(vars_xgboost_to_load, library='pd')
print(f"4. Reco Final (recoFtr) cargado: {len(recoFtr)} eventos")



1. Gen Non-Filtered (genNFtr) cargado: 11589148 eventos
2. Gen Filtered (genFtr) cargado: 307688 eventos
3. Reco Gen Level Denom (recoGen) cargado: 6298017 eventos
4. Reco Final (recoFtr) cargado: 900424 eventos


In [3]:

recoFtr["q2"] = recoFtr["massJ"]**2 
recoGen["q2Gen"] = recoGen["massJ"]**2  

GenNFlt = genNFtr.copy()     
GenFlt  = genFtr.copy()       

RecoGenFlt = recoGen.copy()             
mask_mass = (recoFtr["massB_test"] > 5.0) & (recoFtr["massB_test"] < 5.6)
Reco = recoFtr[mask_mass].copy()
#2
#eff_Gen, obs_Gen = train_test_split(GenNFlt, test_size=0.3, random_state=22)
eff_Gen, obs_Gen = train_test_split(GenNFlt, test_size=0.03, random_state=549)
eff_GenFtr, obs_GenFtr = train_test_split(GenFlt, test_size=0.1, random_state=2)
eff_RecoGenFtr, obs_RecoGenFtr = train_test_split(RecoGenFlt, test_size=0.3, random_state=2)
eff_RecoFtr, obs_RecoFtr = train_test_split(Reco, test_size=0.1, random_state=2)

a1 = np.array(obs_Gen["gen_cosThetaL"])
a2 = np.array(obs_Gen["gen_cosThetaK"])
a3 = np.array(obs_Gen["gen_phi"])

angles = np.array([a1, a2, a3])
valid_observations_mask = ~np.isnan(angles).any(axis=0)
filtered_data = angles[:, valid_observations_mask]

In [4]:
print("Numero de eventes GEN",len(obs_Gen))
print("Numero de eventes GEN Filtered",len(obs_GenFtr))
print("Numero de eventes Reco Gen filtered",len(obs_RecoGenFtr))
print("Numero de eventes Reco Filtered",len(obs_RecoFtr))


Numero de eventes GEN 347675
Numero de eventes GEN Filtered 30769
Numero de eventes Reco Gen filtered 1889406
Numero de eventes Reco Filtered 90043


# NUEVAS FUNVIONES Y PDF

In [5]:

# PDF transformada usando el método Polar 
class FullAngular_Transformed_PDF(zfit.pdf.BasePDF):
    def __init__(self, obs, raw_FL, raw_S3, raw_r1, alpha1, raw_r2, alpha2, raw_r3, alpha3, name="FullAngular_Transformed_PDF"):
        params = {
            'rFL': raw_FL, 'rS3': raw_S3, 
            'r1': raw_r1, 'a1': alpha1,
            'r2': raw_r2, 'a2': alpha2,
            'r3': raw_r3, 'a3': alpha3
        }
        super().__init__(obs, params, name=name)

    def _get_physical_coeffs(self):
        # FL y S3
        FL = 0.5 * (1.0 + tf.math.tanh(self.params['rFL']))
        FT = 1.0 - FL
        S3 = 0.5 * FT * tf.math.tanh(self.params['rS3'])
    
        # (S9, AFB)
        R1_sq = tf.maximum(0.25 * tf.square(FT) - tf.square(S3), 1e-18)
        R1 = tf.sqrt(R1_sq)
        r1_phys = (R1 / 2.0) * (1.0 + tf.math.tanh(self.params['r1']))
        a1 = self.params['a1']
    
        AFB = 1.5 * tf.math.cos(a1) * r1_phys
        S9 = tf.math.sin(a1) * r1_phys
    
        # (S4, S7)
        R2_sq = tf.maximum(FL * (FT - 2.0 * S3), 1e-18)
        R2 = tf.sqrt(R2_sq)
        r2_phys = (R2 / 2.0) * (1.0 + tf.math.tanh(self.params['r2']))
        a2 = self.params['a2']
    
        S4 = 0.5 * tf.math.cos(a2) * r2_phys
        S7 = tf.math.sin(a2) * r2_phys
    
        # (S5, S8)
        R3_sq = tf.maximum(FL * (FT + 2.0 * S3), 1e-18)
        R3 = tf.sqrt(R3_sq)
        r3_phys = (R3 / 2.0) * (1.0 + tf.math.tanh(self.params['r3']))
        a3 = self.params['a3']
    
        S5 = tf.math.cos(a3) * r3_phys
        S8 = 0.5 * tf.math.sin(a3) * r3_phys
    
        return FL, S3, S9, AFB, S4, S7, S5, S8

    def _unnormalized_pdf(self, x):
        vars_list = z.unstack_x(x)
        cos_l = vars_list[0]
        cos_k = vars_list[1]
        phi   = vars_list[2]
    
        # sin_k = tf.sqrt(tf.maximum(1.0 - cos_k**2, 0.0), 1e-10)
        # sin_l = tf.sqrt(tf.maximum(1.0 - cos_l**2, 0.0),1e-10)     
        sin_k = tf.sqrt(tf.maximum(1.0 - cos_k**2, 1e-10))
        sin_l = tf.sqrt(tf.maximum(1.0 - cos_l**2, 1e-10))   
        sin2_k = sin_k**2
        cos2_k = cos_k**2
        sin2_l = sin_l**2
        cos2l_term = 2.0 * cos_l**2 - 1.0
        sin2l_term = 2.0 * sin_l * cos_l
        sin2k_term = 2.0 * sin_k * cos_k
        cos_phi = tf.cos(phi)
        sin_phi = tf.sin(phi)
        cos2_phi = tf.cos(2.0 * phi)
        sin2_phi = tf.sin(2.0 * phi)

        FL, S3, S9, AFB, S4, S7, S5, S8 = self._get_physical_coeffs()
    
        term1 = 0.75 * (1.0 - FL) * sin2_k
        term2 = FL * cos2_k
        term3 = 0.25 * (1.0 - FL) * sin2_k * cos2l_term
        term4 = -1.0 * FL * cos2_k * cos2l_term
        term5 = S3 * sin2_k * sin2_l * cos2_phi
        term6 = S4 * sin2k_term * sin2l_term * cos_phi
        term7 = S5 * sin2k_term * sin_l * cos_phi
        term8 = (4.0/3.0) * AFB * sin2_k * cos_l
        term9 = S7 * sin2k_term * sin_l * sin_phi
        term10 = S8 * sin2k_term * sin2l_term * sin_phi
        term11 = S9 * sin2_k * sin2_l * sin2_phi
    
        pdf = term1 + term2 + term3 + term4 + term5 + term6 + term7 + term8 + term9 + term10 + term11
        return pdf

    @zfit.supports(norm=True)
    def _integrate(self, limits, norm_range, options=None):
        integral_value = 32.0 * np.pi / 9.0
        return tf.constant([integral_value], dtype=self.dtype)


def get_inverse_values(phys_params):
    """
    Toma los valores físicos (FL, S3, S9, AFB, S4, S7, S5, S8) y devuelve los 
    parámetros iniciales sin restricciones (rFL, rS3, r1, a1, r2, a2, r3, a3).
    """
    FL, S3, S9, AFB, S4, S7, S5, S8 = phys_params
    def atanh(x): return np.arctanh(np.clip(x, -0.9999, 0.9999))

    rFL = atanh(2.0 * FL - 1.0)
    FT = 1.0 - FL
    rS3 = atanh(S3 / (0.5 * FT))

    # Inversa Sector 1 (Transversal)
    x1 = (2.0/3.0) * AFB
    y1 = S9
    r1_phys = np.sqrt(x1**2 + y1**2)
    a1 = np.arctan2(y1, x1)

    R1 = np.sqrt(max(1e-15, 0.25 * FT**2 - S3**2))
    ratio1 = r1_phys / R1 if R1 > 0 else 0
    r1 = atanh(2.0 * ratio1 - 1.0)

    # Inversa Sector 2 (Paralelo)
    x2 = 2.0 * S4
    y2 = S7
    r2_phys = np.sqrt(x2**2 + y2**2)
    a2 = np.arctan2(y2, x2)

    R2 = np.sqrt(max(1e-15, FL * (FT - 2.0 * S3)))
    ratio2 = r2_phys / R2 if R2 > 0 else 0
    r2 = atanh(2.0 * ratio2 - 1.0)

    # Inversa Sector 3 (Perpendicular)
    x3 = S5
    y3 = 2.0 * S8
    r3_phys = np.sqrt(x3**2 + y3**2)
    a3 = np.arctan2(y3, x3)

    R3 = np.sqrt(max(1e-15, FL * (FT + 2.0 * S3)))
    ratio3 = r3_phys / R3 if R3 > 0 else 0
    r3 = atanh(2.0 * ratio3 - 1.0)

    return [rFL, rS3, r1, a1, r2, a2, r3, a3]


def apply_transformation_equations(rFL, rS3, r1, a1, r2, a2, r3, a3):
    """
    Aplica las ecuaciones de transformación del espacio polar TRANSFORMADO al FISICO.
    """
    FL = 0.5 * (1.0 + np.tanh(rFL))
    FT = 1.0 - FL
    S3 = 0.5 * FT * np.tanh(rS3)

    # Sector 1
    R1 = np.sqrt(np.maximum(0.25 * FT**2 - S3**2, 1e-15))
    r1_phys = (R1 / 2.0) * (1.0 + np.tanh(r1))
    AFB = 1.5 * np.cos(a1) * r1_phys
    S9 = np.sin(a1) * r1_phys

    # Sector 2
    R2 = np.sqrt(np.maximum(FL * (FT - 2.0 * S3), 1e-15))
    r2_phys = (R2 / 2.0) * (1.0 + np.tanh(r2))
    S4 = 0.5 * np.cos(a2) * r2_phys
    S7 = np.sin(a2) * r2_phys

    # Sector 3
    R3 = np.sqrt(np.maximum(FL * (FT + 2.0 * S3), 1e-15))
    r3_phys = (R3 / 2.0) * (1.0 + np.tanh(r3))
    S5 = np.cos(a3) * r3_phys
    S8 = 0.5 * np.sin(a3) * r3_phys

    return {'FL': FL, 'S3': S3, 'S9': S9, 'AFB': AFB, 'S4': S4, 'S7': S7, 'S5': S5, 'S8': S8}


def print_polar_physical_variables(rFL, rS3, r1, a1, r2, a2, r3, a3, bin_name=""):
    """
    Calcula e imprime los radios máximos, radios físicos y ángulos 
    reales a partir de los parámetros libres del ajuste.
    """
    #  FL y S3 físicos
    FL = 0.5 * (1.0 + np.tanh(rFL))
    FT = 1.0 - FL
    S3 = 0.5 * FT * np.tanh(rS3)
    
    # calcular los radios
    R1 = np.sqrt(np.maximum(0.25 * FT**2 - S3**2, 1e-15))
    r1_phys = (R1 / 2.0) * (1.0 + np.tanh(r1))
    
    R2 = np.sqrt(np.maximum(FL * (FT - 2.0 * S3), 1e-15))
    r2_phys = (R2 / 2.0) * (1.0 + np.tanh(r2))
    
    R3 = np.sqrt(np.maximum(FL * (FT + 2.0 * S3), 1e-15))
    r3_phys = (R3 / 2.0) * (1.0 + np.tanh(r3))
    
    print("\n" + "="*60)
    print(f"VARIABLES POLARES FÍSICAS (Radios y Ángulos Reales) - {bin_name}")
    print("="*60)
    
    print(f"[Sector 1: S9, AFB]")
    print(f"  Radio máximo (R1)     = {R1:.6f}")
    print(f"  Radio físico (r1_phys)= {r1_phys:.6f}")
    print(f"  Ángulo (alpha_1)      = {a1:.6f} rad")
    
    print(f"\n[Sector 2: S4, S7]")
    print(f"  Radio máximo (R2)     = {R2:.6f}")
    print(f"  Radio físico (r2_phys)= {r2_phys:.6f}")
    print(f"  Ángulo (alpha_2)      = {a2:.6f} rad")
    
    print(f"\n[Sector 3: S5, S8]")
    print(f"  Radio máximo (R3)     = {R3:.6f}")
    print(f"  Radio físico (r3_phys)= {r3_phys:.6f}")
    print(f"  Ángulo (alpha_3)      = {a3:.6f} rad")
    print("="*60 + "\n")


def get_physical_array(r_array):
    phys_dict = apply_transformation_equations(*r_array)
    return np.array([phys_dict.get(key, 0.0) for key in obs_order])


def compute_jacobian(func, x, epsilon=1e-5):
    n_in = len(x)
    y_center = func(x)
    n_out = len(y_center)
    
    J = np.zeros((n_out, n_in))
    
    for i in range(n_in):
        x_plus = np.copy(x)
        x_minus = np.copy(x)
        
        x_plus[i] += epsilon
        x_minus[i] -= epsilon
        
        y_plus = func(x_plus)
        y_minus = func(x_minus)
        
        J[:, i] = (y_plus - y_minus) / (2.0 * epsilon)
        
    return J



# GEN FIT TRANFORMED SPACE (POLAR TRANSFORMATION)

In [6]:
#%%capture data_gen_fit_transformed_polar
#q2_bins = {"bin1":[1.1, 2.0],"bin2": [2.0, 4.0],"bin3":[4.0, 6.0], "bin4":[6.0, 7.0], "bin5":[7.0, 8.0],"bin7":[11.0, 12.5], "bin9":[15.0, 17.0], "bin10":[17.0, 23.0]}
q2_bins = {"bin10":[17.0, 23.0]}

for binN in q2_bins.keys():
    print(f"\n{'='*60}\nProcesando {binN} con rango q2: {q2_bins[binN]}\n{'='*60}")
    
    # init_rFL, init_rS3 = 0.235, -0.167
    # init_r1, init_a1 = -2.210, -1.585
    # init_r2, init_a2 = 0.278, 1
    # init_r3, init_a3 = -1.160, -1.310

    json_path = f"fit_results/gen/gen_phy_slsqp/{binN}/fit_results_gen_physical_slsqp_{binN}.json"
    print(f"-> Leyendo semillas iniciales de: {json_path}")
    with open(json_path, 'r') as file:
        data = json.load(file)
    params_dict = data.get("parameters", {})
    phys_params_list = [params_dict["FL"]["value"], params_dict["S3"]["value"], params_dict["S9"]["value"], params_dict["AFB"]["value"], params_dict["S4"]["value"], params_dict["S7"]["value"], params_dict["S5"]["value"], params_dict["S8"]["value"]]
    inv_vals = get_inverse_values(phys_params_list)
    print(inv_vals)
    init_rFL, init_rS3, init_r1, init_a1, init_r2, init_a2, init_r3, init_a3 = inv_vals
        


    # ======================================================
    # CONFIGURACIÓN DEL ESPACIO 
    # ======================================================
    cos_l = zfit.Space('cos_l', limits=(-1, 1))
    cos_k = zfit.Space('cos_k', limits=(-1, 1))
    phi   = zfit.Space('phi',   limits=(-np.pi, np.pi)) 
    obs_ang = cos_l * cos_k * phi  

    # ======================================================
    # CREACIÓN DE PARÁMETROS ZFIT CON LAS SEMILLAS DINÁMICAS
    # ======================================================
    limit = 8
    # F_L y S3
    rFL = zfit.Parameter(f'rFL_{binN}', init_rFL, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    rS3 = zfit.Parameter(f'rS3_{binN}', init_rS3, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    # S9y AFB 
    r1 = zfit.Parameter(f'rr1_{binN}', init_r1, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    a1 = zfit.Parameter(f'a1_{binN}', init_a1, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    # S4 y S7 
    r2 = zfit.Parameter(f'rr2_{binN}', init_r2, step_size=1e-3, lower_limit=-limit, upper_limit=100)
    a2 = zfit.Parameter(f'a2_{binN}', init_a2, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    # 3 S5 y S8 
    r3 = zfit.Parameter(f'rr3_{binN}', init_r3, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    a3 = zfit.Parameter(f'a3_{binN}', init_a3, step_size=1e-3, lower_limit=-limit, upper_limit=limit)

    # # F_L y S3
    # rFL = zfit.Parameter(f'rFL_{binN}', init_rFL, step_size=1e-3)
    # rS3 = zfit.Parameter(f'rS3_{binN}', init_rS3, step_size=1e-3)
    # # S9y AFB 
    # r1 = zfit.Parameter(f'rr1_{binN}', init_r1, step_size=1e-3)
    # a1 = zfit.Parameter(f'a1_{binN}', init_a1, step_size=1e-3)
    # # S4 y S7 
    # r2 = zfit.Parameter(f'rr2_{binN}', init_r2, step_size=1e-3)
    # a2 = zfit.Parameter(f'a2_{binN}', init_a2, step_size=1e-3)
    # # 3 S5 y S8 
    # r3 = zfit.Parameter(f'rr3_{binN}', init_r3, step_size=1e-3)
    # a3 = zfit.Parameter(f'a3_{binN}', init_a3, step_size=1e-3)




    fit_params_list = [rFL, rS3, r1, a1, r2, a2, r3, a3]
    # ======================================================
    # PDFs Y CARGA DE DATOS
    # ======================================================
    
    pdf_ang_trans = FullAngular_Transformed_PDF(obs_ang, rFL, rS3, r1, a1, r2, a2, r3, a3)
    obs_Gen_q2 = select_q2_bin(obs_Gen, binN, "q2Gen")
    print("NÚMERO DE EVENTOS:", len(obs_Gen_q2),"\n")
    data_true = zfit.Data.from_numpy(array=obs_Gen_q2[["gen_cosThetaL", "gen_cosThetaK", "gen_phi"]].to_numpy(), obs=obs_ang)

    # ======================================================
    # FITS
    # ======================================================

    print("\n" + "="*60)
    print(">>> INICIANDO FIT GEN LEVEL TRANSFORMED SPACE <<<")
    print("="*60)
    result_gen, errors_gen = run_fit(pdf_ang_trans, data_true)
    r_values = np.array([result_gen.params[p]['value'] for p in fit_params_list])
    cov_fit = result_gen.covariance(params=fit_params_list)
    print(result_gen.params)
    print_polar_physical_variables(*r_values, bin_name=binN)
    obs_order = ['FL', 'AFB', 'S3', 'S4', 'S5', 'S7', 'S8', 'S9']
    J = compute_jacobian(get_physical_array, r_values)
    cov_phys = J @ cov_fit @ J.T
    errors_phys = np.sqrt(np.diag(cov_phys))

    phys_vals_gen_dict = apply_transformation_equations(*r_values)
    print("\n" + "-"*80)
    print(f"RESUMEN DE OBSERVABLES FÍSICOS (Bin: {binN})")
    print("-"*80)
    print(f"{'Observable':<10} | {'Valor Físico':<15} | {'Error':<25}")
    print("-"*80)

    for i, key in enumerate(obs_order):
        val = phys_vals_gen_dict.get(key, 0.0)
        err = errors_phys[i]
        print(f"{key:<10} | {val:>15.6f} | +/- {err:<15.6f}")
    print("-"*80 + "\n")



    save_fit_results(result_gen, binN, base_dir="fit_results/gen/gen_transformed_polar", name=f"fit_results_gen_transformed_polar_{binN}")    
    save_phy_results(result_zfit=result_gen, phys_dict=phys_vals_gen_dict, cov_phys=cov_phys, obs_order=obs_order, bin_n=binN, base_dir="fit_results/gen/gen_transformed_polar", name=f"fit_results_gen_transformed_polar_phy_{binN}")
    plot_nll_profiles(result_gen, fit_params_list, binN, base_dir=f"fit_results/gen/gen_transformed_polar/{binN}/nll_profiles")
    save_correlation_matrix(result_gen, fit_params_list, binN, out_dir=f"fit_results/gen/gen_transformed_polar/{binN}")




Procesando bin10 con rango q2: [17.0, 23.0]
-> Leyendo semillas iniciales de: fit_results/gen/gen_phy_slsqp/bin10/fit_results_gen_physical_slsqp_bin10.json
[-0.3269980996351908, -0.8140721810945593, -1.9607865140274776, 2.8783582822731395, 3.01981838935972, -3.1376386264649114, -1.8024130685629265, -0.1294552333723227]
NÚMERO DE EVENTOS: 18827 


>>> INICIANDO FIT GEN LEVEL TRANSFORMED SPACE <<<


2026-03-16 23:40:01.743235: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-16 23:40:01.793741: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-16 23:40:01.795909: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

name         value  (rounded)                minos    at limit
---------  ------------------  -------------------  ----------
rFL_bin10           -0.325079  -  0.011   +  0.011       False
rS3_bin10           -0.816607  -  0.042   +   0.04       False
rr1_bin10            -1.93999  -      3   +   0.46       False
a1_bin10            -0.820216  -    1.4   +    2.1       False
rr2_bin10              1.9435  -    0.4   +   0.59       False
a2_bin10             -3.14419  -  0.013   +  0.013       False
rr3_bin10            -1.78427  -    1.1   +   0.43       False
a3_bin10             -2.91984  -    1.5   +    1.2       False

VARIABLES POLARES FÍSICAS (Radios y Ángulos Reales) - bin10
[Sector 1: S9, AFB]
  Radio máximo (R1)     = 0.242924
  Radio físico (r1_phys)= 0.004915
  Ángulo (alpha_1)      = -0.820216 rad

[Sector 2: S4, S7]
  Radio máximo (R2)     = 0.614034
  Radio físico (r2_phys)= 0.601695
  Ángulo (alpha_2)      = -3.144193 rad

[Sector 3: S5, S8]
  Radio máximo (R3)     = 0.2